In [ ]:
import numpy as np
import pandas as pd
from numba import njit
import seaborn as sns
import matplotlib.pyplot as plt

# Get rid of annoying tf warning
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import bayesflow as bf
import tensorflow as tf

from tensorflow.keras.layers import LSTM, Bidirectional
from tensorflow.keras.models import Sequential

In [ ]:
%load_ext autoreload
%autoreload 2
# Suppress scientific notation for floats
np.set_printoptions(suppress=True)
# Configure rng
RNG = np.random.default_rng()

In [ ]:
# gpu setting and checking
physical_devices = tf.config.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(physical_devices[0], enable=True)
print(tf.config.list_physical_devices('GPU'))

In [ ]:
TRAIN_NETWORK = True

In [ ]:
# plotting
THETA_NAMES = ("Learning rate", "Sensitivity", "Perseverance")
THETA_LABELS = (r"$\alpha$", r"$\tau$", r"$\kappa$")

FONT_SIZE_1 = 18
FONT_SIZE_2 = 16
FONT_SIZE_3 = 12

## Helpers

In [ ]:
@njit
def repeat_rows_randomly(arr):
    # Pre-allocate a large enough array
    max_possible_repeats = arr.shape[0] * 202
    repeated_array = np.empty((max_possible_repeats, arr.shape[1]), dtype=arr.dtype)
    idx = 0
    for i in range(arr.shape[0]):
        # Sample a random number of repeats between 6 and 202
        n_repeats = np.random.randint(6, 202)
        # Repeat the current row n times and add to the pre-allocated array
        for _ in range(n_repeats):
            repeated_array[idx] = arr[i]
            idx += 1
    # Slice the array to keep only the filled portion
    return repeated_array[:idx]

@njit
def softmax(x, tau):
    e_x = np.exp(x * tau)
    out = e_x / e_x.sum()
    return out

@njit
def select_action(p):
    x = np.arange(2)
    return x[np.searchsorted(np.cumsum(p), np.random.random(), side="right")]

## Context

In [ ]:
@njit
def generate_context():
    num_blocks = 12
    conditions = np.array(
        [
            [0.25, 0.05], [0.125, 0.05], [0.08, 0.05],
            [0.05, 0.25], [0.05, 0.125], [0.05, 0.08],
            [0.25, 0.05], [0.125, 0.05], [0.08, 0.05],
            [0.05, 0.25], [0.05, 0.125], [0.05, 0.08]
        ]
    )
    # Create block idx
    numbers_column = np.arange(0, num_blocks).reshape(num_blocks, 1)
    # Shuffle the conditions
    idx = np.random.choice(np.arange(num_blocks), num_blocks, replace=False)
    shuffled_conditions = conditions[idx]
    # Repeat each row with different randomly drawn n
    out = repeat_rows_randomly(
        np.hstack((numbers_column, shuffled_conditions))
    )
    return out

## Prior

In [ ]:
@njit
def sample_theta_0():
    alpha = np.random.beta(a=1.5, b=2)
    tau = np.random.beta(a=1.1, b=2.5) * 15
    kappa = np.random.uniform(low=-0.5, high=0.5)
    return np.array([alpha, tau, kappa])

@njit
def sample_eta():
    alpha_std = np.abs(np.random.normal(0, 0.01))
    tau_std = np.abs(np.random.normal(0, 0.2))
    kappa_std = np.abs(np.random.normal(0, 0.01))
    return np.array([alpha_std, tau_std, kappa_std])

@njit
def sample_theta_t(eta, num_steps):
    lower_bounds = np.array([0, 0, -0.5])
    upper_bounds = np.array([1, 15, 0.5])
    theta_t = np.zeros((num_steps, 3))
    theta_t[0] = sample_theta_0()
    z = np.random.randn(num_steps - 1, 3)
    for t in range(1, num_steps):
        theta_t[t] = np.maximum(
            np.minimum(theta_t[t-1] + eta * z[t-1], upper_bounds),
            lower_bounds
        )
    return theta_t.astype(np.float32)

In [ ]:
context = generate_context()
block_idx = context.shape[0]
eta = sample_eta()
theta = sample_theta_t(eta, block_idx)
time = np.arange(theta.shape[0])
fig, axarr = plt.subplots(1, 3, figsize=(12, 3))
for i, ax in enumerate(axarr.flat):
    ax.grid(alpha=0.5)
    ax.plot(
        time,
        theta[:, i],
        color='maroon'
    )
    ax.set_title(f'{THETA_NAMES[i]} ({THETA_LABELS[i]})', fontsize=FONT_SIZE_1)
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_3)
    if i == 0:
        ax.set_ylabel("Parameter value", fontsize=FONT_SIZE_2)
    ax.set_xlabel("Time step", fontsize=FONT_SIZE_2)

sns.despine()
fig.tight_layout()

In [ ]:
LOCAL_PRIOR_MEAN = np.array([0.444, 5.453 , -0.001])
LOCAL_PRIOR_STD = np.array([0.26, 3.873 , 0.29])
GLOBAL_PRIOR_MEAN = np.array([0.008, 0.16 , 0.008])
GLOBAL_PRIOR_STD = np.array([0.006, 0.12 , 0.006])

## Likelihood

In [ ]:
@njit
def sample_softmax_rl(theta, context):
    num_steps = context.shape[0]
    sim_data = np.zeros((num_steps, 2))
    values = np.full(2, 0.0)
    kappa = np.zeros(2)
    for t in range(num_steps):
        if t != 0 and context[t, 0] != context[t-1, 0]:
            values = np.full(2, 0.0)
        action_probs = softmax(values + kappa, theta[t, 1])
        resp = select_action(action_probs)
        sim_data[t, 0] = resp
        sim_data[t, 1] = np.random.binomial(1, context[t, int(resp)+1])
        values[int(resp)] += theta[t, 0] * (sim_data[t, 1] - values[int(resp)])
        kappa = np.zeros(2)
        kappa[resp] = theta[t, 2]
    return sim_data

## Generative Model

In [ ]:
def generative_model(batch_size):
    # generate non-batchable context
    context = generate_context()
    num_steps = context.shape[0]
    # data containers
    eta = np.zeros((batch_size, 3))
    theta = np.zeros((batch_size, num_steps, 3))
    sim_data = np.zeros((batch_size, num_steps, 3))
    # data generation
    for i in range(batch_size):
        eta[i] = sample_eta()
        theta[i] = sample_theta_t(eta[i], num_steps)
        sim_data[i, :, :2] = sample_softmax_rl(theta[i], context)
        sim_data[i, :, 2] = context[:, 0] / 12
    # data configuration
    out = {
        "local_parameters": ((theta - LOCAL_PRIOR_MEAN) / LOCAL_PRIOR_STD).astype(np.float32),
        "hyper_parameters":((eta - GLOBAL_PRIOR_MEAN) / GLOBAL_PRIOR_STD).astype(np.float32),
        "summary_conditions": sim_data.astype(np.float32),
        "direct_conditions": np.log(num_steps) * np.ones((batch_size, 1), dtype=np.float32)
    }
    return out

In [ ]:
def configure_input(forward_dict):
    return forward_dict

## Neural Approximator

In [ ]:
approximator_settings = {
    "lstm1_hidden_units": 512,
    "lstm2_hidden_units": 256,
    "lstm3_hidden_units": 128,
    "trainer": {
        "max_to_keep": 1,
        "default_lr": 5e-4,
        "memory": False,
    }
}

In [ ]:
summary_network = bf.networks.HierarchicalNetwork(
    [
        Sequential(
            [
                Bidirectional(LSTM(approximator_settings["lstm1_hidden_units"], return_sequences=True)),
                Bidirectional(LSTM(approximator_settings["lstm2_hidden_units"], return_sequences=True)),
            ]
        ),
        Sequential(
            [
                Bidirectional(LSTM(approximator_settings["lstm3_hidden_units"]))
            ]
        )
    ]
)

In [ ]:
local_network = bf.amortizers.AmortizedPosterior(
    bf.networks.InvertibleNetwork(
        num_params=3,
        num_coupling_layers=8,
        coupling_settings={
            "dense_args": dict(kernel_regularizer=None),
            "dropout": False,
            "coupling_design": 'interleaved'
        }
    )
)
global_network = bf.amortizers.AmortizedPosterior(
    bf.networks.InvertibleNetwork(
        num_params=3,
        num_coupling_layers=6,
        coupling_settings={
            "dense_args": dict(kernel_regularizer=None),
            "dropout": False,
            "coupling_design": 'interleaved'
        }
    )
)

In [ ]:
amortizer = bf.amortizers.TwoLevelAmortizedPosterior(
    local_amortizer=local_network,
    global_amortizer=global_network,
    summary_net=summary_network
)
trainer = bf.trainers.Trainer(
    amortizer=amortizer,
    generative_model=generative_model,
    configurator=configure_input,
    **approximator_settings.get("trainer"),
    checkpoint_path="checkpoints/rl_kappa"
)

## Training

In [ ]:
if TRAIN_NETWORK:
    history = trainer.train_online(
        epochs=200,
        iterations_per_epoch=2000,
        batch_size=64
    )
else:
    history = trainer.loss_history.get_plottable()